# TF-IDF Sweep

### Load the Processed Tables
- The splits, the normalised tags and the shared vocabulary all come from the preparation notebook
- The bag of words summary is loaded too, because it is the figure this sweep has to beat

In [1]:
from pathlib import Path

import pandas as pd

candidate_paths = [
    Path("data/processed"),
    Path("../data/processed"),
]
processed_dir = next((path for path in candidate_paths if path.exists()), None)
if processed_dir is None:
    raise FileNotFoundError(
        "Could not find data/processed. Run data_preparation.ipynb first."
    )
baseline_path = processed_dir / "nb_bow_summary.csv"
if not baseline_path.exists():
    raise FileNotFoundError(
        "Could not find nb_bow_summary.csv. Run bag_of_words_baseline.ipynb first."
    )
splits = pd.read_csv(processed_dir / "splits.csv", dtype={"id": str})
tags_frame = pd.read_csv(processed_dir / "tags_normalised.csv", dtype={"id": str})
vocabulary_frame = pd.read_csv(processed_dir / "shared_vocabulary_50.csv")
baseline = pd.read_csv(baseline_path)

tag_vocabulary = list(vocabulary_frame["tag"])

print("Splits shape:", splits.shape)
print("Tags in the shared vocabulary:", len(tag_vocabulary))
print("Bag of words micro F1 to beat:", baseline.set_index("split")["micro_f1"].to_dict())
baseline

Splits shape: (82810, 4)
Tags in the shared vocabulary: 50
Bag of words micro F1 to beat: {'random': 0.26, 'artist': 0.239}


,split,terms,micro_precision,micro_recall,micro_f1,macro_f1,true_positive,false_positive,false_negative,fit_seconds
0,random,86194,0.233,0.293,0.260,0.152,8960,29442,21647,2.0
1,artist,85307,0.221,0.260,0.239,0.131,8026,28216,22856,2.0


### Read the Lyrics Back
- The same two cleaning functions as the preparation notebook, so the tokens match
- The sha1 fingerprints and the word counts are checked against lyrics_clean.csv, which confirms each document sits on the row its id belongs to

In [2]:
import hashlib
import re

candidate_paths = [
    Path("music4all-en"),
    Path("../music4all-en"),
]
data_dir = next((path for path in candidate_paths if path.exists()), None)
if data_dir is None:
    raise FileNotFoundError(
        "Could not find music4all-en. Unzip the subset to the repo root."
    )
lyrics_dir = data_dir / "lyrics"
marker_pattern = re.compile(r"^\s*\[[^\]]{1,30}\]\s*$")


def clean_lines(text):
    kept = []
    for line in text.replace("\r\n", "\n").replace("\r", "\n").split("\n"):
        line = line.rstrip()
        if not line.strip():
            continue
        if marker_pattern.match(line):
            continue
        kept.append(line)

    return "\n".join(kept)


def flatten(text):
    return re.sub(r"\s+", " ", clean_lines(text).lower()).strip()


track_ids = list(splits["id"])
documents = []
for track_id in track_ids:
    path = lyrics_dir / (track_id + ".txt")
    raw = path.read_text(encoding="utf-8", errors="replace")
    documents.append(flatten(raw))

word_counts = pd.Series([len(text.split()) for text in documents], index=track_ids)
fingerprints = pd.Series(
    [hashlib.sha1(text.encode("utf-8")).hexdigest() for text in documents],
    index=track_ids,
)
expected = pd.read_csv(processed_dir / "lyrics_clean.csv", dtype={"id": str}).set_index("id")

print("Documents read:", len(documents))
print("Mean words:", round(word_counts.mean(), 1))
print("Fingerprints matching lyrics_clean.csv:",
      int((fingerprints == expected.loc[track_ids, "fingerprint"]).sum()), "of", len(track_ids))
print("Word counts matching lyrics_clean.csv:",
      int((word_counts == expected.loc[track_ids, "n_words"]).sum()), "of", len(track_ids))

Documents read: 82810
Mean words: 242.9
Fingerprints matching lyrics_clean.csv: 82810 of 82810
Word counts matching lyrics_clean.csv: 82810 of 82810


### Build the Target Matrix
- One binary column per tag in the shared vocabulary, one row per track, in the order of splits.csv
- Identical to the baseline notebook, so the two sets of scores are comparable

In [3]:
import numpy as np

tag_lists = {}
for record in tags_frame.itertuples(index=False):
    if isinstance(record.tags, str) and record.tags != "":
        tag_lists[record.id] = record.tags.split(",")
    else:
        tag_lists[record.id] = []

position = {}
for i in range(len(tag_vocabulary)):
    position[tag_vocabulary[i]] = i

target = np.zeros((len(track_ids), len(tag_vocabulary)), dtype=np.int8)
row = 0
for track_id in track_ids:
    for tag in tag_lists[track_id]:
        if tag in position:
            target[row, position[tag]] = 1
    row = row + 1

print("Target shape:", target.shape)
print("Cardinality:", round(float(target.sum(axis=1).mean()), 2))
print("Smallest support over all rows:", int(target.sum(axis=0).min()))

Target shape: (82810, 50)
Cardinality: 2.46
Smallest support over all rows: 1175


### Index the Two Splits
- Row positions rather than ids, so the feature matrix and the target can be sliced the same way
- The test rows are counted here but never fed to a model, because everything is selected on validation

In [4]:
split_rows = {}
for name in ["random", "artist"]:
    column = splits[name + "_split"]
    parts = {}
    for part in ["train", "valid", "test"]:
        parts[part] = np.flatnonzero(column == part)
    split_rows[name] = parts

    print(name, "split:",
          "train", len(parts["train"]),
          "valid", len(parts["valid"]),
          "test", len(parts["test"]))

random split: train 57966 valid 12421 test 12423
artist split: train 57997 valid 12424 test 12389


### Why the Decision Threshold Has to Be Swept Too
- TF-IDF weights are normalised, so a document carries far less total feature mass than raw counts do
- Multinomial Naive Bayes scales its log likelihood with that mass, so the class prior decides almost every tag
- predict applies an implicit cut at 0.5, which suits counts and not TF-IDF, so a config judged at 0.5 is judged under a broken rule
- The cut is therefore swept alongside the vectoriser parameters and every config is scored at its own best cut

### The Sweep Grid
- min_df drops terms that appear in too few training documents, max_df drops terms that appear in too many
- The n-gram range decides whether word pairs are counted alongside single words
- Every combination is scored at every threshold, so the table shows the spread and not only the winner

In [5]:
import itertools

min_df_values = [1, 2, 5, 10, 20]
max_df_values = [0.5, 0.9, 1.0]
ngram_values = [(1, 1), (1, 2)]
threshold_values = [0.5, 0.3, 0.2, 0.15, 0.1, 0.05, 0.02]

configurations = []
for ngram_range, min_df, max_df in itertools.product(ngram_values, min_df_values, max_df_values):
    configurations.append({"ngram_range": ngram_range, "min_df": min_df, "max_df": max_df})

print("Configurations per split:", len(configurations))
print("Thresholds per configuration:", len(threshold_values))
print("Rows the sweep will produce:",
      len(configurations) * len(threshold_values) * len(split_rows))
print("Naive Bayes models to train:", len(configurations) * len(split_rows) * len(tag_vocabulary))

Configurations per split: 30
Thresholds per configuration: 7
Rows the sweep will produce: 420
Naive Bayes models to train: 3000


### Run the Sweep
- One TfidfVectorizer per configuration, fitted on the training rows only
- The classifier is the same per tag multinomial Naive Bayes the baseline used, so only the features change
- predict_proba is taken once per tag and then cut at each threshold, which costs no extra training

In [6]:
import time

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import precision_recall_fscore_support
from sklearn.naive_bayes import MultinomialNB

documents = np.array(documents, dtype=object)

sweep_rows = []
mass_rows = []
for name in ["random", "artist"]:
    parts = split_rows[name]
    y_train = target[parts["train"]]
    y_valid = target[parts["valid"]]
    counts = CountVectorizer()
    counts_mass = float(counts.fit_transform(documents[parts["train"]]).sum(axis=1).mean())
    for configuration in configurations:
        started = time.perf_counter()
        vectoriser = TfidfVectorizer(
            ngram_range=configuration["ngram_range"],
            min_df=configuration["min_df"],
            max_df=configuration["max_df"],
        )
        X_train = vectoriser.fit_transform(documents[parts["train"]])
        X_valid = vectoriser.transform(documents[parts["valid"]])
        scores = np.zeros((X_valid.shape[0], len(tag_vocabulary)))
        for j in range(len(tag_vocabulary)):
            model = MultinomialNB()
            model.fit(X_train, y_train[:, j])
            scores[:, j] = model.predict_proba(X_valid)[:, 1]

        elapsed = time.perf_counter() - started
        mass_rows.append({
            "split": name,
            "ngram_range": str(configuration["ngram_range"]),
            "min_df": configuration["min_df"],
            "max_df": configuration["max_df"],
            "terms": len(vectoriser.vocabulary_),
            "counts_mass_per_document": round(counts_mass, 1),
            "tfidf_mass_per_document": round(float(X_train.sum(axis=1).mean()), 2),
        })
        for threshold in threshold_values:
            predicted = (scores >= threshold).astype(np.int8)
            micro = precision_recall_fscore_support(
                y_valid, predicted, average="micro", zero_division=0,
            )
            macro = precision_recall_fscore_support(
                y_valid, predicted, average="macro", zero_division=0,
            )
            sweep_rows.append({
                "split": name,
                "ngram_range": str(configuration["ngram_range"]),
                "min_df": configuration["min_df"],
                "max_df": configuration["max_df"],
                "threshold": threshold,
                "terms": len(vectoriser.vocabulary_),
                "micro_precision": round(micro[0], 3),
                "micro_recall": round(micro[1], 3),
                "micro_f1": round(micro[2], 3),
                "macro_f1": round(macro[2], 3),
                "predicted": int(predicted.sum()),
            })
        best_here = max(
            [r for r in sweep_rows
             if r["split"] == name
             and r["ngram_range"] == str(configuration["ngram_range"])
             and r["min_df"] == configuration["min_df"]
             and r["max_df"] == configuration["max_df"]],
            key=lambda r: r["micro_f1"],
        )

        print(name,
              "ngram", configuration["ngram_range"],
              "min_df", configuration["min_df"],
              "max_df", configuration["max_df"],
              "terms", len(vectoriser.vocabulary_),
              "best micro F1", best_here["micro_f1"],
              "at threshold", best_here["threshold"],
              "seconds", round(elapsed, 1))
sweep = pd.DataFrame(sweep_rows)
mass = pd.DataFrame(mass_rows)

print()
print("Sweep finished, rows:", len(sweep))

random ngram (1, 1) min_df 1 max_df 0.5 terms 86175 best micro F1 0.216 at threshold 0.02 seconds 6.6


random ngram (1, 1) min_df 1 max_df 0.9 terms 86193 best micro F1 0.209 at threshold 0.02 seconds 6.6


random ngram (1, 1) min_df 1 max_df 1.0 terms 86194 best micro F1 0.208 at threshold 0.02 seconds 6.6


random ngram (1, 1) min_df 2 max_df 0.5 terms 42000 best micro F1 0.236 at threshold 0.02 seconds 6.3


random ngram (1, 1) min_df 2 max_df 0.9 terms 42018 best micro F1 0.236 at threshold 0.02 seconds 6.4


random ngram (1, 1) min_df 2 max_df 1.0 terms 42019 best micro F1 0.236 at threshold 0.02 seconds 6.4


... (50 outputs left out)

artist ngram (1, 2) min_df 10 max_df 1.0 terms 100106 best micro F1 0.252 at threshold 0.02 seconds 15.2


artist ngram (1, 2) min_df 20 max_df 0.5 terms 54169 best micro F1 0.267 at threshold 0.05 seconds 14.5


artist ngram (1, 2) min_df 20 max_df 0.9 terms 54187 best micro F1 0.266 at threshold 0.05 seconds 14.7


artist ngram (1, 2) min_df 20 max_df 1.0 terms 54188 best micro F1 0.266 at threshold 0.05 seconds 14.8

Sweep finished, rows: 420


### The Feature Mass Gap
- This is the quantity that makes the default cut unusable for TF-IDF
- Raw counts give a document around 230 units of mass, TF-IDF gives it between 6 and 11

In [7]:
summary_mass = mass.groupby("split")[
    ["counts_mass_per_document", "tfidf_mass_per_document"]
].agg(["mean", "min", "max"]).round(2)

print("Mass per document, counts against TF-IDF:")
display(summary_mass)

print("Ratio at the lightest and heaviest TF-IDF configuration:")
for name in ["random", "artist"]:
    rows = mass[mass["split"] == name]
    print(" ", name,
          "counts", rows["counts_mass_per_document"].iloc[0],
          "TF-IDF from", rows["tfidf_mass_per_document"].min(),
          "to", rows["tfidf_mass_per_document"].max(),
          "so a factor of",
          round(rows["counts_mass_per_document"].iloc[0] / rows["tfidf_mass_per_document"].max(), 1),
          "at best")

Mass per document, counts against TF-IDF:


counts_mass_per_document               tfidf_mass_per_document        \
                           mean    min    max                    mean   min   
split                                                                         
artist                    231.8  231.8  231.8                    8.02  5.91   
random                    230.0  230.0  230.0                    8.01  5.90   

               
          max  
split          
artist  10.68  
random  10.67

Ratio at the lightest and heaviest TF-IDF configuration:
  random counts 230.0 TF-IDF from 5.9 to 10.67 so a factor of 21.6 at best
  artist counts 231.8 TF-IDF from 5.91 to 10.68 so a factor of 21.7 at best


### What the Default Cut Would Have Concluded
- Scoring every config at 0.5 only, which is what predict does, picks the unigram range instead of the bigram one
- This is why the threshold is swept jointly rather than fixed

In [8]:
for name in ["random", "artist"]:
    at_default = sweep[(sweep["split"] == name) & (sweep["threshold"] == 0.5)]
    at_default = at_default.sort_values("micro_f1", ascending=False)
    jointly = sweep[sweep["split"] == name].sort_values("micro_f1", ascending=False)
    naive = at_default.iloc[0]
    honest = jointly.iloc[0]

    print(name, "split")
    print("  judged at 0.5 only : ngram", naive["ngram_range"],
          "min_df", naive["min_df"], "max_df", naive["max_df"],
          "micro F1", naive["micro_f1"], "macro F1", naive["macro_f1"])
    print("  judged jointly     : ngram", honest["ngram_range"],
          "min_df", honest["min_df"], "max_df", honest["max_df"],
          "threshold", honest["threshold"],
          "micro F1", honest["micro_f1"], "macro F1", honest["macro_f1"])

random split
  judged at 0.5 only : ngram (1, 1) min_df 20 max_df 0.5 micro F1 0.044 macro F1 0.036
  judged jointly     : ngram (1, 2) min_df 20 max_df 0.5 threshold 0.05 micro F1 0.273 macro F1 0.084
artist split
  judged at 0.5 only : ngram (1, 1) min_df 20 max_df 0.5 micro F1 0.036 macro F1 0.034
  judged jointly     : ngram (1, 2) min_df 20 max_df 0.5 threshold 0.05 micro F1 0.267 macro F1 0.079


### The Grid, Split by Split
- Each configuration reduced to its best threshold, sorted by micro F1

In [9]:
best_per_configuration = {}
for name in ["random", "artist"]:
    rows = sweep[sweep["split"] == name]
    index = rows.groupby(["ngram_range", "min_df", "max_df"])["micro_f1"].idxmax()
    table = rows.loc[index].sort_values("micro_f1", ascending=False)
    best_per_configuration[name] = table

    print(name, "split, best five configurations at their best threshold:")
    display(table.head(5))
    print(name, "split, worst three:")
    display(table.tail(3))

random split, best five configurations at their best threshold:


,split,ngram_range,min_df,max_df,threshold,terms,micro_precision,micro_recall,micro_f1,macro_f1,predicted
194,random,"(1, 2)",20,0.5,0.05,53828,0.231,0.334,0.273,0.084,44235
87,random,"(1, 1)",20,0.5,0.15,10143,0.262,0.282,0.272,0.094,32939
201,random,"(1, 2)",20,0.9,0.05,53846,0.234,0.325,0.272,0.080,42464
208,random,"(1, 2)",20,1.0,0.05,53847,0.235,0.323,0.272,0.080,42086
101,random,"(1, 1)",20,1.0,0.15,10162,0.262,0.277,0.269,0.091,32258


random split, worst three:


,split,ngram_range,min_df,max_df,threshold,terms,micro_precision,micro_recall,micro_f1,macro_f1,predicted
111,random,"(1, 2)",1,0.5,0.02,1636816,0.258,0.008,0.015,0.003,890
118,random,"(1, 2)",1,0.9,0.02,1636834,0.250,0.006,0.012,0.003,752
125,random,"(1, 2)",1,1.0,0.02,1636835,0.249,0.006,0.012,0.003,734


artist split, best five configurations at their best threshold:


,split,ngram_range,min_df,max_df,threshold,terms,micro_precision,micro_recall,micro_f1,macro_f1,predicted
404,artist,"(1, 2)",20,0.5,0.05,54169,0.225,0.327,0.267,0.079,44775
411,artist,"(1, 2)",20,0.9,0.05,54187,0.229,0.318,0.266,0.076,43004
418,artist,"(1, 2)",20,1.0,0.05,54188,0.229,0.317,0.266,0.075,42695
297,artist,"(1, 1)",20,0.5,0.15,10254,0.258,0.271,0.264,0.085,32381
311,artist,"(1, 1)",20,1.0,0.15,10273,0.260,0.266,0.263,0.082,31633


artist split, worst three:


,split,ngram_range,min_df,max_df,threshold,terms,micro_precision,micro_recall,micro_f1,macro_f1,predicted
321,artist,"(1, 2)",1,0.5,0.02,1639452,0.252,0.007,0.013,0.003,806
328,artist,"(1, 2)",1,0.9,0.02,1639470,0.239,0.005,0.010,0.002,682
335,artist,"(1, 2)",1,1.0,0.02,1639471,0.239,0.005,0.010,0.002,661


### What Each Parameter Does on Its Own
- The mean best micro F1 over every configuration holding one parameter fixed
- This separates a parameter that matters from one that happens to sit next to a good value

In [10]:
for parameter in ["ngram_range", "min_df", "max_df", "threshold"]:
    if parameter == "threshold":
        grouped = sweep.groupby(["split", parameter])["micro_f1"].agg(["mean", "max"])
    else:
        frames = []
        for name in ["random", "artist"]:
            frames.append(best_per_configuration[name])
        combined = pd.concat(frames)
        grouped = combined.groupby(["split", parameter])["micro_f1"].agg(["mean", "max", "min"])
    print("Effect of", parameter)
    display(grouped.round(3))

Effect of ngram_range


mean    max    min
split  ngram_range                     
artist (1, 1)       0.242  0.264  0.215
       (1, 2)       0.159  0.267  0.010
random (1, 1)       0.244  0.272  0.208
       (1, 2)       0.163  0.273  0.012

Effect of min_df


mean    max    min
split  min_df                     
artist 1       0.114  0.221  0.010
       2       0.143  0.236  0.048
       5       0.226  0.242  0.208
       10      0.252  0.254  0.252
       20      0.265  0.267  0.263
random 1       0.112  0.216  0.012
       2       0.147  0.236  0.055
       5       0.229  0.245  0.211
       10      0.258  0.260  0.257
       20      0.271  0.273  0.269

Effect of max_df


mean    max    min
split  max_df                     
artist 0.5     0.203  0.267  0.013
       0.9     0.199  0.266  0.010
       1.0     0.199  0.266  0.010
random 0.5     0.206  0.273  0.015
       0.9     0.202  0.272  0.012
       1.0     0.202  0.272  0.012

Effect of threshold


mean    max
split  threshold              
artist 0.02       0.183  0.253
       0.05       0.172  0.267
       0.10       0.145  0.259
       0.15       0.117  0.264
       0.20       0.086  0.241
       0.30       0.037  0.116
       0.50       0.011  0.036
random 0.02       0.187  0.259
       0.05       0.175  0.273
       0.10       0.149  0.268
       0.15       0.121  0.272
       0.20       0.092  0.247
       0.30       0.042  0.131
       0.50       0.015  0.044

### Compare Against the Bag of Words Baseline
- The baseline is plain word counts with every vectoriser parameter left at its default
- Two TF-IDF figures are reported, one at the default cut and one at the jointly selected cut
- Only the second is a fair comparison, because the default cut penalises TF-IDF for its different scale

In [11]:
comparison_rows = []
for name in ["random", "artist"]:
    reference = baseline[baseline["split"] == name].iloc[0]
    at_default = sweep[(sweep["split"] == name) & (sweep["threshold"] == 0.5)]
    naive = at_default.sort_values("micro_f1", ascending=False).iloc[0]
    best = sweep[sweep["split"] == name].sort_values("micro_f1", ascending=False).iloc[0]
    comparison_rows.append({
        "split": name,
        "bow_micro_f1": reference["micro_f1"],
        "tfidf_micro_f1_default_cut": naive["micro_f1"],
        "tfidf_micro_f1_tuned_cut": best["micro_f1"],
        "micro_gain": round(best["micro_f1"] - reference["micro_f1"], 3),
        "bow_macro_f1": reference["macro_f1"],
        "tfidf_macro_f1_tuned_cut": best["macro_f1"],
        "macro_gain": round(best["macro_f1"] - reference["macro_f1"], 3),
        "best_threshold": best["threshold"],
        "best_ngram_range": best["ngram_range"],
        "best_min_df": best["min_df"],
        "best_max_df": best["max_df"],
        "bow_terms": int(reference["terms"]),
        "tfidf_terms": int(best["terms"]),
    })

    print(name, "split:",
          "bag of words", reference["micro_f1"],
          "TF-IDF at the default cut", naive["micro_f1"],
          "TF-IDF at a cut of", best["threshold"], "gives", best["micro_f1"])
comparison = pd.DataFrame(comparison_rows)

display(comparison)

random split: bag of words 0.26 TF-IDF at the default cut 0.044 TF-IDF at a cut of 0.05 gives 0.273
artist split: bag of words 0.239 TF-IDF at the default cut 0.036 TF-IDF at a cut of 0.05 gives 0.267


,split,bow_micro_f1,tfidf_micro_f1_default_cut,tfidf_micro_f1_tuned_cut,micro_gain,bow_macro_f1,tfidf_macro_f1_tuned_cut,macro_gain,best_threshold,best_ngram_range,best_min_df,best_max_df,bow_terms,tfidf_terms
0,random,0.260,0.044,0.273,0.013,0.152,0.084,-0.068,0.05,"(1, 2)",20,0.5,86194,53828
1,artist,0.239,0.036,0.267,0.028,0.131,0.079,-0.052,0.05,"(1, 2)",20,0.5,85307,54169


### Per Tag Scores at the Winning Configuration
- The winning configuration and threshold are refitted so the per tag table matches the baseline's shape

In [12]:
tag_tables = {}
for name in ["random", "artist"]:
    parts = split_rows[name]
    best = sweep[sweep["split"] == name].sort_values("micro_f1", ascending=False).iloc[0]
    vectoriser = TfidfVectorizer(
        ngram_range=eval(best["ngram_range"]),
        min_df=int(best["min_df"]),
        max_df=float(best["max_df"]),
    )
    X_train = vectoriser.fit_transform(documents[parts["train"]])
    X_valid = vectoriser.transform(documents[parts["valid"]])
    y_train = target[parts["train"]]
    y_valid = target[parts["valid"]]
    scores = np.zeros((X_valid.shape[0], len(tag_vocabulary)))
    for j in range(len(tag_vocabulary)):
        model = MultinomialNB()
        model.fit(X_train, y_train[:, j])
        scores[:, j] = model.predict_proba(X_valid)[:, 1]

    predicted = (scores >= best["threshold"]).astype(np.int8)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_valid, predicted, average=None, zero_division=0,
    )
    tag_tables[name] = pd.DataFrame({
        "tag": tag_vocabulary,
        "support": support,
        "predicted": predicted.sum(axis=0),
        "precision": precision.round(3),
        "recall": recall.round(3),
        "f1": f1.round(3),
    })
best_tags = tag_tables["random"].sort_values("f1", ascending=False)

print("Best five tags at the winning configuration:")
display(best_tags.head(5))
print("Worst five tags at the winning configuration:")
display(best_tags.tail(5))
print("Tags above F1 0.3:", int((best_tags["f1"] > 0.3).sum()))
print("Tags below F1 0.1:", int((best_tags["f1"] < 0.1).sum()))
print("Tags never predicted:", int((best_tags["predicted"] == 0).sum()))

Best five tags at the winning configuration:


,tag,support,predicted,precision,recall,f1
16,hip-hop,594,484,0.663,0.540,0.596
19,rap,455,343,0.682,0.514,0.586
0,rock,3161,12058,0.262,0.998,0.415
8,metal,852,1062,0.348,0.434,0.387
1,pop,2362,9415,0.240,0.958,0.384


Worst five tags at the winning configuration:


,tag,support,predicted,precision,recall,f1
43,psychedelic,229,0,0.0,0.0,0.0
45,lo-fi,235,0,0.0,0.0,0.0
44,pop rock,200,0,0.0,0.0,0.0
46,fip,189,0,0.0,0.0,0.0
48,shoegaze,189,0,0.0,0.0,0.0


Tags above F1 0.3: 7
Tags below F1 0.1: 40
Tags never predicted: 9


### Compare the Per Tag Scores Against the Baseline
- Tag by tag, so it is clear whether TF-IDF helps the frequent tags, the rare ones, or both

In [13]:
movement_rows = []
for name in ["random", "artist"]:
    reference = pd.read_csv(processed_dir / ("nb_bow_per_tag_" + name + ".csv"))
    merged = reference[["tag", "support", "f1"]].merge(
        tag_tables[name][["tag", "f1"]], on="tag", suffixes=("_bow", "_tfidf"),
    )
    merged["change"] = (merged["f1_tfidf"] - merged["f1_bow"]).round(3)
    movement_rows.append(merged.assign(split=name))

    print(name, "split:",
          "tags improved", int((merged["change"] > 0).sum()),
          "tags worsened", int((merged["change"] < 0).sum()),
          "mean change", round(float(merged["change"].mean()), 3))
movement = pd.concat(movement_rows)
random_movement = movement[movement["split"] == "random"].sort_values("change")

print("Tags TF-IDF hurts most:")
display(random_movement.head(5))
print("Tags TF-IDF helps most:")
display(random_movement.tail(5))

random split: tags improved 4 tags worsened 46 mean change -0.068
artist split: tags improved 4 tags worsened 44 mean change -0.052
Tags TF-IDF hurts most:


,tag,support,f1_bow,f1_tfidf,change,split
18,rnb,449,0.234,0.004,-0.230,random
9,soul,663,0.259,0.042,-0.217,random
36,heavy metal,343,0.221,0.021,-0.200,random
15,dance,532,0.267,0.070,-0.197,random
32,metalcore,289,0.188,0.000,-0.188,random


Tags TF-IDF helps most:


,tag,support,f1_bow,f1_tfidf,change,split
45,lo-fi,235,0.006,0.000,-0.006,random
8,metal,852,0.373,0.387,0.014,random
19,rap,455,0.557,0.586,0.029,random
16,hip-hop,594,0.559,0.596,0.037,random
3,alternative,1636,0.214,0.257,0.043,random


### Save

In [14]:
sweep.to_csv(processed_dir / "tfidf_sweep.csv", index=False)

print("Wrote:", processed_dir / "tfidf_sweep.csv")

comparison.to_csv(processed_dir / "tfidf_vs_bow.csv", index=False)

print("Wrote:", processed_dir / "tfidf_vs_bow.csv")

mass.to_csv(processed_dir / "tfidf_feature_mass.csv", index=False)

print("Wrote:", processed_dir / "tfidf_feature_mass.csv")

movement.to_csv(processed_dir / "tfidf_per_tag_change.csv", index=False)

print("Wrote:", processed_dir / "tfidf_per_tag_change.csv")

for name in ["random", "artist"]:
    out_path = processed_dir / ("tfidf_per_tag_" + name + ".csv")
    tag_tables[name].to_csv(out_path, index=False)

    print("Wrote:", out_path)

comparison

Wrote: data\processed\tfidf_sweep.csv
Wrote: data\processed\tfidf_vs_bow.csv
Wrote: data\processed\tfidf_feature_mass.csv
Wrote: data\processed\tfidf_per_tag_change.csv
Wrote: data\processed\tfidf_per_tag_random.csv
Wrote: data\processed\tfidf_per_tag_artist.csv


,split,bow_micro_f1,tfidf_micro_f1_default_cut,tfidf_micro_f1_tuned_cut,micro_gain,bow_macro_f1,tfidf_macro_f1_tuned_cut,macro_gain,best_threshold,best_ngram_range,best_min_df,best_max_df,bow_terms,tfidf_terms
0,random,0.260,0.044,0.273,0.013,0.152,0.084,-0.068,0.05,"(1, 2)",20,0.5,86194,53828
1,artist,0.239,0.036,0.267,0.028,0.131,0.079,-0.052,0.05,"(1, 2)",20,0.5,85307,54169
